In [1]:
import pandas as pd
import numpy as np
import random
import pyomo.environ as pyo
from pyomo.environ import *
from pyomo.environ import SolverFactory
import yfinance as yf
import matplotlib.pyplot as plt


In [4]:
preco_venda = pd.read_excel('ercerc11.xlsx')
custos = pd.read_excel('ercerc11.xlsx', sheet_name='custos')
tempo = pd.read_excel('ercerc11.xlsx',sheet_name='tempo')

In [9]:
preco_venda=preco_venda.set_index('brinquedo')
custos=custos.set_index('brinquedo')
tempo=tempo.set_index('brinquedo')

In [51]:
custo_total = custos.T.sum()

In [55]:
tempo

,pintura,carpintaria
brinquedo,,
soldado,2,1
trem,1,1


In [57]:
tempo_maximo ={
    'pintura':100,
    'carpintaria':80
}

In [ ]:
model = pyo.ConcreteModel()

model.brinquedos = pyo.Set(initialize=preco_venda.index)
model.servicos = pyo.Set(initialize=tempo.columns)
model.custos = pyo.Set(initialize=custos.columns)

model.x = pyo.Var(model.brinquedos,bounds=(0,None), domain=pyo.NonNegativeIntegers)

def objetivo(model):

    return sum(
        preco_venda['vendido_por'][b]*model.x[b] - custo_total[b]*model.x[b] for b in model.brinquedos 
    )
model.obj = pyo.Objective(rule=objetivo, sense=pyo.maximize)

def restricao_horas(model, s):

    return sum(
        model.x[b] * tempo[s][b] for b in model.brinquedos
    ) <= tempo_maximo[s]
model.restricao_horas = pyo.Constraint(model.servicos, rule=restricao_horas)

def demanda_maxima_soldado(model):
    return model.x['soldado'] <=40
model.demanda_maxima_soldado = pyo.Constraint(rule=demanda_maxima_soldado)


In [61]:
model.pprint()

3 Set Declarations
    brinquedos : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    2 : {'soldado', 'trem'}
    custos : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    2 : {'materia_prima', 'mo_ci'}
    servicos : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    2 : {'pintura', 'carpintaria'}

1 Var Declarations
    x : Size=2, Index=brinquedos
        Key     : Lower : Value : Upper : Fixed : Stale : Domain
        soldado :     0 :  None :  None : False :  True : NonNegativeIntegers
           trem :     0 :  None :  None : False :  True : NonNegativeIntegers

1 Objective Declarations
    obj : Size=1, Index=None, Active=True
        Key  : Active : Sense    : Expression
        None :   True : maximize : 27*x[soldado] - 24*x[soldado] + 21*x[trem] - 19*x[trem]

2 Constrai

In [68]:
opt = SolverFactory('cplex')
resultado = opt.solve(model, tee=True)


Welcome to IBM(R) ILOG(R) CPLEX(R) Interactive Optimizer 22.1.1.0
  with Simplex, Mixed Integer & Barrier Optimizers
5725-A06 5725-A29 5724-Y48 5724-Y49 5724-Y54 5724-Y55 5655-Y21
Copyright IBM Corp. 1988, 2022.  All Rights Reserved.

Type 'help' for a list of available commands.
Type 'help' followed by a command name for more
information on commands.

CPLEX> Logfile 'cplex.log' closed.
Logfile 'C:\Users\DECIV\AppData\Local\Temp\tmp9xo1cyne.cplex.log' open.
CPLEX> Problem 'C:\Users\DECIV\AppData\Local\Temp\tmpfk_g8s08.pyomo.lp' read.
Read time = 0.00 sec. (0.00 ticks)
CPLEX> Problem name         : C:\Users\DECIV\AppData\Local\Temp\tmpfk_g8s08.pyomo.lp
Objective sense      : Maximize
Variables            :       2  [General Integer: 2]
Objective nonzeros   :       2
Linear constraints   :       3  [Less: 3]
  Nonzeros           :       5
  RHS nonzeros       :       3

Variables            : Min LB: 0.000000         Max UB: all infinite   
Objective nonzeros   : Min   : 2.000000       

In [71]:
for b in model.brinquedos:
    print(f"Quantidade de {b}: ", pyo.value(model.x[b]))
print('Lucro total semanal: ', pyo.value(model.obj))

Quantidade de soldado:  20.0
Quantidade de trem:  60.0
Lucro total semanal:  180.0
